## Work with GPU

In [1]:
import torch
import pandas as pd
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, Dataset

In [2]:
torch.cuda.is_available()

True

In [3]:
import torch

print(torch.__version__)
print(torch.version.cuda)
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

2.11.0+cu128
12.8
True
NVIDIA GeForce RTX 4050 Laptop GPU


In [4]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device being used:',device)

Device being used: cuda


In [5]:
device

device(type='cuda')

In [6]:
df = pd.read_csv("fashion-mnist_train.csv")
df.shape

(60000, 785)

In [7]:
df.head()

,label,pixel1,pixel2,pixel3,pixel4,pixel5,pixel6,pixel7,pixel8,pixel9,...,pixel775,pixel776,pixel777,pixel778,pixel779,pixel780,pixel781,pixel782,pixel783,pixel784
0,2,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,9,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,6,0,0,0,0,0,0,0,5,0,...,0,0,0,30,43,0,0,0,0,0
3,0,0,0,0,1,2,0,0,0,0,...,3,0,0,0,0,1,0,0,0,0
4,3,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [8]:
features = df.iloc[:,1:]
targets = df.iloc[:,0]
targets

0        2
1        9
2        6
3        0
4        3
        ..
59995    9
59996    1
59997    8
59998    8
59999    7
Name: label, Length: 60000, dtype: int64

In [9]:
x_train, x_test, y_train, y_test = train_test_split(features, targets, test_size=0.2, random_state= 42)
x_train /= 255.0
x_test /= 255.0

In [10]:
x_train.shape

(48000, 784)

In [11]:
x_test.shape

(12000, 784)

In [12]:
device

device(type='cuda')

In [13]:
# to create the dataset
class CustomDataset(Dataset):
    def __init__(self, features, targets):
        self.features = torch.tensor(features.to_numpy()).to(torch.float32).to(device)
        self.targets = torch.tensor(targets.to_numpy()).to(torch.long).to(device)
    
    def __len__(self):
        return self.features.shape[0]
    
    def __getitem__(self, index):
        return (self.features[index], self.targets[index])

In [14]:
# dataset collection stored here
train_dataset = CustomDataset(x_train, y_train)
test_dataset = CustomDataset(x_test, y_test)

In [15]:
train_dataset.__len__()

48000

In [16]:
test_dataset.__len__()

12000

In [17]:
train_dataset.__getitem__(0)

(tensor([0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.2275,
         0.5333, 0.0000, 0.0

In [18]:
train_dataset_loader = DataLoader(train_dataset, batch_size= 32, shuffle= True)
test_dataset_loader = DataLoader(test_dataset, batch_size= 32, shuffle=False)

In [19]:
for feature, target in train_dataset_loader:
    print(feature)
    print(target)
    break

tensor([[0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.],
        ...,
        [0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.]], device='cuda:0')
tensor([5, 6, 1, 5, 4, 5, 6, 2, 6, 2, 1, 8, 5, 2, 5, 8, 3, 7, 9, 9, 1, 1, 4, 8,
        4, 7, 6, 1, 8, 8, 4, 5], device='cuda:0')


In [30]:
class ImageClassifier(nn.Module):

    def __init__(self, number_of_features):
        super().__init__()

        # Image classifier model
        self.model = nn.Sequential(
            nn.Linear(number_of_features, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 10)
        )
        # optimizer
        self.optim = optim.SGD(self.model.parameters(), lr = 0.1)
    
    def forward(self, features):
        self.y_pred = self.model(features)
        return self.y_pred
    
    def loss_function(self, y_real):
        loss_fn = nn.CrossEntropyLoss()
        self.loss = loss_fn(self.y_pred, y_real)
        return self.loss.item()
    
    def optimization(self):
        self.optim.zero_grad()
        self.loss.backward()
        self.optim.step()


In [31]:
x_train.shape[1]

784

In [32]:
# model is defined but the model uses the "cpu"
model = ImageClassifier(x_train.shape[1])

In [33]:
device

device(type='cuda')

In [34]:
# making model use gpu instead of cpu
model = model.to(device)

In [35]:
model

ImageClassifier(
  (model): Sequential(
    (0): Linear(in_features=784, out_features=128, bias=True)
    (1): ReLU()
    (2): Linear(in_features=128, out_features=64, bias=True)
    (3): ReLU()
    (4): Linear(in_features=64, out_features=32, bias=True)
    (5): ReLU()
    (6): Linear(in_features=32, out_features=10, bias=True)
  )
)

In [36]:
epochs = 30
total_loss = 0
for epoch in range(epochs):
    epoch += 1
    total_loss = 0
    for features_batch, targets_batch in train_dataset_loader:
        y_pred = model(features_batch)
        loss = model.loss_function(targets_batch)
        model.optimization()
        total_loss = total_loss + loss

    print("epoch:", epoch, "\tloss:", total_loss/len(train_dataset_loader))

epoch: 1 	loss: 0.7285077996551991
epoch: 2 	loss: 0.4484677180300156
epoch: 3 	loss: 0.3964523615737756
epoch: 4 	loss: 0.3650704154620568
epoch: 5 	loss: 0.3441766646256049
epoch: 6 	loss: 0.32780304717024167
epoch: 7 	loss: 0.3165193635722001
epoch: 8 	loss: 0.30135039167602856
epoch: 9 	loss: 0.2921529056703051
epoch: 10 	loss: 0.2809984355221192
epoch: 11 	loss: 0.27198574170966944
epoch: 12 	loss: 0.26362680218617124
epoch: 13 	loss: 0.25582841950903334
epoch: 14 	loss: 0.2507601511950294
epoch: 15 	loss: 0.24351648583697777
epoch: 16 	loss: 0.23560013571195304
epoch: 17 	loss: 0.2316894096074005
epoch: 18 	loss: 0.22799718162355323
epoch: 19 	loss: 0.22091518018518885
epoch: 20 	loss: 0.21570969251170755
epoch: 21 	loss: 0.21278692290683587
epoch: 22 	loss: 0.20953771917894481
epoch: 23 	loss: 0.20190344994328915
epoch: 24 	loss: 0.20206544034151983
epoch: 25 	loss: 0.19916576190665364
epoch: 26 	loss: 0.19213610898206632
epoch: 27 	loss: 0.18543188885599374
epoch: 28 	loss: 0.1

In [37]:
# model ready for evaluation
model.eval()

ImageClassifier(
  (model): Sequential(
    (0): Linear(in_features=784, out_features=128, bias=True)
    (1): ReLU()
    (2): Linear(in_features=128, out_features=64, bias=True)
    (3): ReLU()
    (4): Linear(in_features=64, out_features=32, bias=True)
    (5): ReLU()
    (6): Linear(in_features=32, out_features=10, bias=True)
  )
)

### Accuracy:
We need to find the accurcy now. The accuracy of the data is given by the formula:

Accuracy = (number of correct predictions) / (total number of the predictions)

In [38]:
len(test_dataset_loader)*32

12000

In [39]:
correct = 0
data_records = 0
with torch.no_grad():
    for features_batch, targets_batch in test_dataset_loader:
        y_pred = model(features_batch)
        correct = correct + (targets_batch == torch.max(y_pred, dim = 1)[1]).sum().item()
        data_records = data_records + features_batch.shape[0]
    print("total data_records:", data_records, "correct records:", correct)
    accuracy = correct / data_records
    print(accuracy)

total data_records: 12000 correct records: 10549
0.8790833333333333
